<a href="https://colab.research.google.com/github/kalakar85/2025-github-ur-copilot-workshop/blob/main/Financial%20prompt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# MASTER PROMPT â€” START

You are my product engineer inside ChatGPT Work. Build and publish a complete, private, mobile-friendly personal financial dashboard named **Ledgerly**. This is a generic personal-finance product and is not associated with any business.

Do the work, not merely describe a plan or produce a mockup. Use **ChatGPT Sites** for the application and its server-side APIs. Use the connected **Google Drive** app for a dedicated import folder. Create a recurring **ChatGPT Work automation** that reads that folder every day at 8:00 AM in my local timezone and sends only new data to the Site.

Follow this specification exactly. Do not omit a page, control, persistence behavior, empty state, import path, or verification step. Do not seed sample financial data.

## 1. Non-negotiable product rules

1. Create exactly one Site for this project. Reuse and update it throughout the build; do not create extra copies on follow-up turns.
2. The Site must be private/owner-only unless I explicitly request different access later.
3. Store durable structured data in a Cloudflare D1 database bound to the Site as `DB`.
4. Store original uploaded or Drive-imported file bytes in a Cloudflare R2 bucket bound to the Site as `BUCKET`.
5. Do not use browser local storage as the source of truth. Data must be available from any device after I sign into the same ChatGPT account and open the published Site.
6. The Site itself must not attempt to browse Google Drive from client-side code. The ChatGPT Work automation is the secure bridge: it reads Drive through the authorized connector and posts imports to the Site's protected server endpoint.
7. Create or reuse one dedicated Google Drive folder exactly once. Use the exact folder name `Ledgerly Financial Inbox`. If a folder with that exact name already exists in my Drive, reuse it instead of creating another one.
8. Schedule the Drive import exactly once per day at 8:00 AM in my local timezone. Do not create duplicate schedules. If my timezone is unavailable, ask one concise question for it before creating the schedule.
9. On first launch, all financial datasets must be empty. Do not insert demo/sample transactions, balances, budgets, goals, subscriptions, recurring bills, rules, tags, receipts, statements, invoices, or documents.
10. Starter category names and account-picker names are configuration definitions only, not financial records. They must not contain balances or create transactions.
11. Every visible button, menu, tab, icon, filter, form, and modal described below must work. No placeholder controls and no broken icon glyphs.
12. Use `lucide-react` icons or another installed vector icon library. Do not use unsupported font-icon characters.
13. Make all body and helper text readable. Normal body text should be approximately 14â€“16 px; helper text and table metadata should not be smaller than 12 px. Form controls and touch targets should be at least 44 px high on mobile where practical.
14. Never expose access tokens, account numbers, financial document contents, or secrets in logs, source code, notifications, or chat responses.

## 2. Build and setup sequence

Execute the work in this order so the integrations point to the correct resources:

1. Verify that the Sites and Google Drive connections are available using harmless read-only checks. If either connector asks me to Connect or Reconnect, stop only for that authorization and continue immediately afterward.
2. Search my Drive for a folder whose name is exactly `Ledgerly Financial Inbox`.
3. If exactly one matching folder exists, reuse it. If none exists, create it once. If multiple exact matches exist, show me the matches and ask which one to use; do not create another folder.
4. Record the selected folder's ID, name, and URL for later use. Do not expose its ID as a secret; it is configuration, but do not hard-code someone else's folder ID.
5. Build the Site, D1 schema, R2 binding, pages, APIs, and empty states in this prompt.
6. Publish one private checkpoint deployment and capture its exact URL/slug.
7. Confirm the Site's `/api/drive-sync` endpoint is live before creating the schedule.
8. Create one daily 8:00 AM automation using the exact Site URL and exact Drive folder selected in this setup. If an automation already exists for the same Site and folder, update/reuse it instead of creating a duplicate.
9. Perform one safe manual test using an empty folder listing or a tiny non-sensitive test file only if needed. Do not create sample financial records for the test. Remove any temporary test artifact you created.
10. Verify the published Site on desktop and mobile widths, then report the Site link, Drive folder link, schedule time/timezone, storage configuration, and test results.

## 3. Empty-start contract

The first real user session must start with these values:

- `transactions = []`
- `documents = []`
- `goals = []`
- `budgets = []`
- `subscriptions = []`
- `recurring = []`
- `rules = []`
- `tags = []`
- `dismissedPatterns = []`
- `selectedPeriod = "all-time"`
- assets total = `0`, liabilities total = `0`, but the Net Worth card must display **Not set** until the user explicitly saves asset or liability totals
- all charts display polished empty states instead of invented data
- totals derived from transactions display `$0.00`
- no fake "recent activity," "upcoming payment," insights, or trend percentages

You may create these starter category definitions because they are lookup configuration, not financial data:

`Housing, Groceries, Shopping, Dining, Transportation, Utilities, Subscriptions, Insurance, Health, Entertainment, Income, Needs review, Other`

You may create these starter account names as lookup configuration only, with no balances:

`Main Checking, Everyday Visa, Rewards Card, Cash`

The user must be able to remove any of those definitions in Settings. If you choose to initialize the account list as empty instead, disable the account selector with a clear **Add an account in Settings** action until an account exists. Never create balances automatically.

## 4. Technical architecture and durable storage

Use a React/Vinext-compatible Sites application with server routes. Configure `.openai/hosting.json` with these logical bindings:

- D1 database binding: `DB`
- R2 object bucket binding: `BUCKET`

Use D1 for structured data and metadata. Use R2 only for original uploaded/imported file bytes. The browser should fetch state from server APIs and update the server after every edit.

### 4.1 D1 tables

Create the schema idempotently so deployments and requests do not destroy existing data.

#### `transactions`

- `id` TEXT PRIMARY KEY
- `date` TEXT NOT NULL, ISO `YYYY-MM-DD`
- `merchant` TEXT NOT NULL
- `category` TEXT NOT NULL DEFAULT `Needs review`
- `amount` REAL NOT NULL and stored as a positive magnitude
- `type` TEXT NOT NULL with allowed values `expense` or `income`
- `account` TEXT NOT NULL DEFAULT `Imported account`
- `tags` TEXT NOT NULL DEFAULT `[]`, stored as a JSON array
- `receipt` INTEGER NOT NULL DEFAULT `0`
- `source` TEXT NOT NULL, such as `manual`, `csv`, `document`, or `google-drive`
- `fingerprint` TEXT NOT NULL UNIQUE
- `createdAt` TEXT NOT NULL, ISO timestamp

#### `tags`

- `name` TEXT PRIMARY KEY
- `createdAt` TEXT NOT NULL

#### `rules`

- `id` TEXT PRIMARY KEY
- `whenText` TEXT NOT NULL
- `thenText` TEXT NOT NULL
- `enabled` INTEGER NOT NULL DEFAULT `1`
- `createdAt` TEXT NOT NULL

#### `settings`

- `key` TEXT PRIMARY KEY
- `value` TEXT NOT NULL, JSON or scalar text as appropriate
- `updatedAt` TEXT NOT NULL

Use settings keys for categories, accounts, goals, budgets, subscriptions, recurring payments, dismissed recurring-pattern keys, assets total, liabilities total, whether net worth was explicitly configured, the persisted global date period `selectedPeriod`, Drive folder metadata, Drive sync metadata, processed Drive file IDs, `driveResetAt`, and `freshStart`.

#### `documents`

- `id` TEXT PRIMARY KEY
- `filename` TEXT NOT NULL
- `mimeType` TEXT NOT NULL
- `size` INTEGER NOT NULL
- `objectKey` TEXT NOT NULL UNIQUE
- `status` TEXT NOT NULL with values such as `queued`, `stored`, or `review`
- `source` TEXT NOT NULL, such as `upload` or `google-drive`
- `createdAt` TEXT NOT NULL

### 4.2 R2 object storage

- Store original receipt, invoice, statement, spreadsheet, image, PDF, and other supported document bytes.
- Enforce a maximum file size of 20 MB per file.
- For manual uploads, use a safe object key such as `uploads/<uuid>-<safe-filename>`.
- For Drive imports, use `drive-inbox/<safe-file-id>-<safe-filename>`.
- Add Drive file ID and modified time as object metadata where supported.
- Do not expose raw bucket URLs publicly.

### 4.3 State API

Implement `GET /api/state` to return one normalized JSON payload containing:

- up to 5,000 transactions, newest first
- all tags
- all rules
- decoded settings
- up to 100 document metadata rows, newest first

Never return original file bytes through the state endpoint.

### 4.4 Transaction API

Implement `/api/transactions`:

- `POST` accepts one transaction or a batch.
- Validate a non-empty merchant/source, a valid date, a positive finite amount, and `type` of `expense` or `income`.
- Normalize tags by trimming, removing blanks, and deduplicating case-insensitively.
- Build this duplicate fingerprint before insertion:

`date + "|" + merchant.trim().toLowerCase() + "|" + amount.toFixed(2) + "|" + account.trim().toLowerCase()`

- Rely on the unique fingerprint plus a pre-check to prevent duplicates.
- Return inserted and duplicate counts clearly for a batch.
- `PATCH` updates a transaction's category and/or tags by ID and returns the saved row.
- `DELETE` deletes a transaction by exact ID.

Apply enabled categorization rules only after duplicate detection. Rules must not cause a duplicate to be inserted.

### 4.5 Preferences API

Implement `PUT /api/preferences` to save managed categories, accounts, tags, rules, goals, budgets, subscriptions, recurring entries, dismissed pattern keys, asset/liability totals, and other settings. Validate and normalize all arrays. Never reset unrelated settings when one preference group is updated.

### 4.6 Document API

Implement `POST /api/documents` as multipart upload:

- accept one or multiple supported files
- reject files over 20 MB with a readable error
- write original bytes to R2
- write document metadata to D1
- use status `queued` until parsed, `stored` after successful storage/processing, and `review` when extraction is uncertain
- never create a financial transaction from uncertain values

### 4.7 Complete data wipe API

Implement `DELETE /api/state`. It must require the JSON confirmation value exactly:

`DELETE ALL LEDGERLY DATA`

On a valid request:

1. Delete all transactions, documents, rules, tags, and settings rows from D1.
2. Delete every object belonging to this Ledgerly Site from R2.
3. Recreate only the structural empty-state settings needed for the app.
4. Save `freshStart = true`.
5. Save `driveResetAt` as the current ISO timestamp.
6. Save assets and liabilities as zero and `netWorthConfigured = false`.
7. Reset `selectedPeriod = "all-time"` so the freshly wiped Site opens in **All time**.
8. Return a clear success payload.

The wipe does not delete files from Google Drive. The daily automation must read `driveResetAt` and never reimport a Drive file whose modified time is at or before that timestamp. This lets the user start fresh without old inbox files repopulating the Site.

## 5. Global application shell and visual system

Build a calm, polished financial interface:

- light gray application background
- white cards with subtle borders and soft shadows
- primary violet accent around `#6558D3`
- green for positive income/savings states
- orange for spending or caution states
- blue for secondary savings/information states
- dark navy summary panels where contrast is appropriate
- rounded cards around 14â€“16 px radius
- desktop left sidebar approximately 238 px wide
- desktop sticky top bar approximately 76 px high
- generous whitespace and consistent alignment

Desktop navigation appears in the sidebar. Mobile uses a compact top bar and a horizontally scrollable bottom navigation that can reach every tab. Do not hide later tabs off-screen without scrolling affordance.

The navigation order must be:

1. Dashboard
2. Transactions
3. Recurring
4. Subscriptions
5. Budgets
6. Goals
7. Documents
8. Rules
9. Settings

The global top bar must include these common actions:

- **Drive sync**
- **Import**
- **Add entry**

All dialogs must trap focus, close with a visible close button and Escape, have labels/ARIA attributes, and remain usable on narrow phones. Show inline success/error feedback and disable submit buttons while saving.

## 6. Dashboard page

### 6.1 Date period

Provide a working period selector with:

- All time
- This month
- Last month
- Last 3 months
- Last 6 months
- This year

Use **All time** on the first-ever visit and whenever no valid saved period exists. `All time` means there is no start-date cutoff: include every saved transaction through the present.

Persist the global period under the D1 settings key `selectedPeriod`. Allowed values are `all-time`, `this-month`, `last-month`, `last-3-months`, `last-6-months`, and `this-year`. When the user selects a period:

1. Recalculate all affected views immediately.
2. Save the new value through `PUT /api/preferences` without resetting any other preference.
3. Keep that value after refresh, sign-out/sign-in, opening the Site on another device, or starting a later session.
4. On every launch, load the saved value from `GET /api/state` before rendering date-dependent totals. Avoid briefly showing another period while state loads.
5. If saving fails, restore the previously saved selection and show a clear error.

The selected period must actually recalculate date-dependent totals, charts, recent activity, and transaction views. Do not make it a decorative button. The Dashboard and Transactions pages must share the same persisted selection, so changing it on either page updates the other.

### 6.2 Summary cards

Show four cards across on wide desktop and responsively stack them on smaller screens:

1. **Net Worth**
   - Formula: `total assets - total liabilities`.
      - Assets and liabilities are user-entered in Settings; transaction cash flow does not automatically become net worth.
         - Until the user explicitly saves totals, display `Not set`, explanatory text, and a link/button to Settings.
            - After setup, show the calculated currency value.
            2. **Income**
               - Sum all `income` transactions in the selected period.
               3. **Spending**
                  - Sum all `expense` transactions in the selected period.
                  4. **Savings rate**
                     - Formula: `((income - spending) / income) * 100`.
                        - If income is zero, show `0%` and avoid division errors.

                        The small areas at the bottom of these cards must not be unexplained decorative boxes. Use a labeled calculation strip or a real compact trend derived from saved data. If there is not enough history, show a readable `No trend yet` state.

                        Do not invent month-over-month arrows or percentages. Show a comparison only when both current and prior-period real data exist.

                        ### 6.3 Dashboard content

                        - **Cash flow chart:** a responsive line/area chart using saved dated transactions. Provide up to seven monthly points. Income is violet/green and expenses are neutral gray/orange. If empty, show `Import or add transactions to see cash flow.`
                        - **Spending by category:** donut/pie chart grouped from real expense transactions for the selected period, with labels, accessible legend, values, and percentages. If empty, show an empty state.
                        - **Recent activity:** five newest transactions in the selected period with merchant, date, category, account, and signed amount.
                        - **Ledgerly insight:** show useful, factual insights only, such as a count of transactions in `Needs review`. Do not generate fake savings advice from absent data.
                        - **Coming up:** show confirmed recurring expenses/subscriptions due soon. If none, say so and link to Recurring.

                        ## 7. Transactions page

                        Provide a responsive transaction table/list with:

                        - search by merchant, category, or tag
                        - account filter
                        - category filter
                        - shared working date-period selector
                        - columns/fields: date and merchant, category, account, tags, amount
                        - income formatted with a leading plus and positive color
                        - expenses formatted with a leading minus and standard/dark color
                        - receipt-matched indicator when applicable

                        ### 7.1 Inline category editing

                        The category cell for every transaction must be directly editable. Tapping/clicking it opens a dropdown of the current managed categories. Save immediately to D1 through `PATCH /api/transactions`, update the row without a full reload, and show an error if persistence fails.

                        ### 7.2 Inline tag editing

                        - Render tags as removable pills.
                        - Tapping a pill removes that tag after saving.
                        - Place a visible **+** control at the end of each row's tags.
                        - The **+** opens a tag-only modal. It must not ask the user to create or select a category.
                        - The modal lets the user select one or more existing tags and optionally create one simple new tag by typing only the tag name.
                        - Creating a tag must not require a category, rule, color, or other metadata.
                        - Save both the global tag definition and the transaction's updated tag array.

                        ### 7.3 Add entry

                        The global **Add entry** button opens a modal with:

                        - segmented Expense / Income selection
                        - amount, initially blank
                        - merchant or source, initially blank
                        - date, defaulted to today but editable
                        - category selector from managed categories
                        - account selector from managed accounts
                        - tags selector with simple add-new-tag support
                        - checkbox `I have a receipt to attach`
                        - optional file picker shown when the receipt box is enabled

                        The form must never prefill a fake amount or merchant. Validate before save, persist the transaction, optionally persist the receipt, close on success, and refresh all affected summaries.

                        ## 8. Imports and duplicate handling

                        ### 8.1 CSV statement import

                        The global **Import** button must support CSV bank or card statements.

                        1. Let the user choose a CSV and preview the detected columns.
                        2. Recognize common headers for date, description/merchant, amount, debit, credit, category, and account.
                        3. If mapping is ambiguous, show a mapping step; do not guess silently.
                        4. Normalize dates to `YYYY-MM-DD`.
                        5. Store the amount as a positive magnitude and map debit/negative rows to `expense`, credit/positive rows to `income`, honoring the statement's actual sign convention.
                        6. Preserve a supported statement category; otherwise use `Needs review`.
                        7. Save `source = csv` and `receipt = false`.
                        8. Run the same fingerprint duplicate detector as every other path.
                        9. Present an import result containing inserted, duplicate, skipped, and needs-review counts.
                        10. Never create placeholder rows for unparseable lines.

                        ### 8.2 Manual document import

                        The Import flow and Documents page must also accept receipts, invoices, images, PDFs, spreadsheets, and other supported documents:

                        - store the original file in R2 first
                        - extract merchant/payee, date, total, and type only when grounded in the document
                        - categorize only when supported; otherwise use `Needs review`
                        - set `receipt = true` for a receipt/invoice-backed transaction
                        - flag uncertain extractions as `review` instead of inventing values
                        - use the same duplicate fingerprint before inserting any extracted transaction

                        ### 8.3 Duplicate rules

                        Duplicate detection must be centralized and apply to manual entry, CSV, documents, and Drive. Use the transaction fingerprint plus the unique database constraint. A duplicate must not be inserted twice even if two imports happen simultaneously. Keep the original document metadata when appropriate, but clearly report that its transaction was a duplicate.

                        ## 9. Automatic recurring and subscripti

SyntaxError: unterminated string literal (detected at line 16) (3463417945.py, line 16)